In [21]:
import os
import sys
import django

# Add project root to path and configure Django
sys.path.insert(0, os.path.dirname(os.getcwd()))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'settings.settings')
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = 'true'
django.setup()

In [22]:
import pandas as pd
from jobs.models import Job, ExtendedJob

## 1. Load Raw Data

In [23]:
# Query jobs that have extended job data using select_related for efficiency
jobs_with_extended = ExtendedJob.objects.select_related('job').values(
    'job__title',
    'job_category',
    'job__company',
    'job__location',
    'job__salary',
    'bachelor_required',
    'master_required',
    'phd_required',
    'tech_stack',
    'min_experience_years',
    'us_only',
    'employment_type',
    'medical_insurance',
)

# Convert to DataFrame
df = pd.DataFrame(list(jobs_with_extended))

# Rename columns to remove the 'job__' prefix
df.columns = [
    'title',
    'job_category',
    'company',
    'location',
    'salary',
    'bachelor_required',
    'master_required',
    'phd_required',
    'tech_stack',
    'min_experience_years',
    'us_only',
    'employment_type',
    'medical_insurance',
]

df

,title,job_category,company,location,salary,bachelor_required,master_required,phd_required,tech_stack,min_experience_years,us_only,employment_type,medical_insurance
0,Service Desk Support Analyst III - Kelsey - Se...,HD,Optum,", US, TXPearland",$28.94 - $51.63 an hour,True,False,False,"windows,active directory,microsoft office,wan,...",4,True,full-time,True
1,Principal Enterprise Architect (Remote),SSWE,IQVIA,", US, NJWayne","$118,100 - $328,800 a year",True,False,False,"Kafka,Apigee,Azure API Management,Kong,Azure M...",15,True,full-time,True
2,Senior Technical Program Manager (ST PM) (15.35),PM,"OCT Consulting, LLC",", US, DCWashington","$175,000 - $250,000 a year",True,False,False,"aws,cloud,cybersecurity,agile,itil,cmmi,pmboK",8,True,full-time,True
3,Cloud Cybersecurity Manager (CCM) (15.35),DevOps,"OCT Consulting, LLC",", US, DCWashington","$150,000 - $225,000 a year",True,False,False,"aws,cloud,cybersecurity,zerotrust,nist,stigs,s...",8,True,full-time,True
4,Cloud Architect and Infrastructure Lead (CAIL)...,DevOps,"OCT Consulting, LLC",", US, DCWashington","$150,000 - $200,000 a year",True,False,False,"aws,devsecops,agile,cybersecurity",6,True,full-time,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1907,"Senior Staff Software Engineer, Backend (Consu...",SSWE,Coinbase,", USRemote","$246,500 - $290,000 a year",True,False,False,"go,temporal,kubernetes,mongodb",10,False,full-time,True
1908,"Staff Software Engineer, Backend (Consumer - T...",SSWE,Coinbase,", USRemote","$218,025 - $256,500 a year",False,False,False,"golang,clickhouse,kafka,redis,mongodb",8,False,full-time,True
1909,Engineering Manager (Consumer - Growth),EM,Coinbase,", USRemote","$218,025 - $256,500 a year",False,False,False,NaN,7,False,full-time,True
1910,"Senior Staff Software Engineer, Backend (Consu...",SSWE,Coinbase,", USRemote","$253,895 - $298,700 a year",False,False,False,NaN,0,False,full-time,True


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1912 entries, 0 to 1911
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   title                 1912 non-null   str   
 1   job_category          1912 non-null   str   
 2   company               1912 non-null   str   
 3   location              1912 non-null   str   
 4   salary                1912 non-null   str   
 5   bachelor_required     1912 non-null   bool  
 6   master_required       1912 non-null   bool  
 7   phd_required          1912 non-null   bool  
 8   tech_stack            1637 non-null   str   
 9   min_experience_years  1912 non-null   int64 
 10  us_only               1912 non-null   bool  
 11  employment_type       1845 non-null   str   
 12  medical_insurance     1907 non-null   object
dtypes: bool(4), int64(1), object(1), str(7)
memory usage: 142.0+ KB


## 2. Clean String Cols

### 2.1 Clean: Salary

In [32]:
print(df['salary'].sample(30).to_string())

1277          $120,000 - $160,000 a year
1617            $72,800 - $91,000 a year
1821            $61,700 - $87,600 a year
1218            $70,000 - $75,000 a year
1749            $75,000 - $90,000 a year
1135          $115,000 - $130,000 a year
751           $130,000 - $160,000 a year
1246          $142,000 - $212,000 a year
99      $138,257.60 - $200,512.00 a year
1291          $109,000 - $191,000 a year
786             $56,000 - $95,000 a year
1879          $102,000 - $178,540 a year
778           $145,000 - $153,000 a year
430           $115,000 - $117,500 a year
1691          $180,000 - $203,000 a year
167           $100,000 - $115,000 a year
484           $110,000 - $130,000 a year
1747          $128,039 - $173,229 a year
1886           $90,000 - $110,000 a year
393           $115,000 - $160,000 a year
1875            $70,000 - $85,000 a year
362           $101,970 - $203,940 a year
893           $307,500 - $397,500 a year
1545           $92,250 - $130,000 a year
437           $1

In [33]:
import re

def clean_salary_to_hourly(salary_str):
    """
    Convert salary string to hourly rate (float).
    - Removes $ and , symbols
    - Averages MIN-MAX ranges
    - Converts monthly (200-40k) and annual (>40k) to hourly
    """
    if pd.isna(salary_str) or not salary_str:
        return None
    
    # Remove $ and , symbols
    cleaned = salary_str.replace('$', '').replace(',', '')
    
    # Extract all numbers (including decimals)
    numbers = re.findall(r'[\d.]+', cleaned)
    
    if not numbers:
        return None
    
    # Convert to floats
    values = [float(n) for n in numbers]
    
    # Average if range (MIN-MAX)
    salary = sum(values) / len(values)
    
    # Convert to hourly based on value thresholds
    if salary < 200:
        # Already hourly
        hourly = salary
    elif salary < 40000:
        # Monthly -> divide by 173.33
        hourly = salary / 173.33
    else:
        # Annual -> divide by 2080
        hourly = salary / 2080
    
    return round(hourly, 2)

# Apply the cleaning function
df['salary_hourly'] = df['salary'].apply(clean_salary_to_hourly)

# Show sample of original vs cleaned
df[['salary', 'salary_hourly']].head(10)

,salary,salary_hourly
0,$28.94 - $51.63 an hour,40.29
1,"$118,100 - $328,800 a year",107.43
2,"$175,000 - $250,000 a year",102.16
3,"$150,000 - $225,000 a year",90.14
4,"$150,000 - $200,000 a year",84.13
5,"$120,001 - $160,000 a year",67.31
6,"$140,000 - $165,000 a year",73.32
7,"$111,000 - $156,500 a year",64.30
8,"$105,000 - $125,000 a year",55.29
9,"$45,000 - $110,000 a year",37.26


In [34]:
df["salary"] = df["salary_hourly"]
df.drop(columns="salary_hourly", inplace=True)

### 2.2 Clean: Employment type

In [35]:
df['employment_type'].unique()

array(['full-time', 'contract', 'flexible', nan, 'internship',
       'part-time'], dtype=object)

In [36]:
# Clean employment_type: keep only the first value if multiple exist
df['employment_type'] = df['employment_type'].str.split(',').str[0]

df['employment_type'].unique()

array(['full-time', 'contract', 'flexible', nan, 'internship',
       'part-time'], dtype=object)

### 2.3 Clean: Location

In [37]:
df['location'].unique()

<StringArray>
[    'US, TX',     'US, NJ',     'US, DC',     'US, CA',     'US, CO',
     'US, VA',     'US, WA',     'US, MN',     'US, UT',     'US, FL',
     'US, GA',     'US, MA',     'US, IL',     'US, OH',     'US, WV',
     'US, AZ',     'US, SC',     'US, NC', 'US, Remote',         'US',
     'US, IN',     'US, MD',     'US, MI',     'US, NY',     'US, KY',
     'US, ME',     'US, NV',     'US, NM',     'US, PA',     'US, OR',
     'US, NH',     'US, CT',     'US, MO',     'US, WI',     'US, OK',
     'US, TN',     'US, KS',     'US, LA',     'US, RI',     'US, NE',
     'US, ND',     'US, IA',     'US, MT',     'US, VT',     'US, ID',
     'US, AL',     'US, WY',     'US, MS']
Length: 48, dtype: str

In [38]:
print(df['location'].sample(30).to_string())

1351        US, CA
1678        US, AZ
750         US, WA
1823        US, VT
977         US, MN
1625    US, Remote
331     US, Remote
1062        US, VA
1694    US, Remote
1424        US, FL
702         US, CO
1396        US, OH
483             US
449     US, Remote
800     US, Remote
262     US, Remote
1177            US
902         US, MT
1096        US, TX
524         US, NC
457     US, Remote
1215    US, Remote
1143    US, Remote
616         US, DC
1408        US, VA
1878        US, NC
245     US, Remote
764         US, AZ
1572        US, DC
969         US, OR


In [39]:
def clean_location(loc_str):
    """
    Clean location to format: "US" or "US, Remote" or "US, STATE"
    
    Input formats:
    - ', US, TXPearland' -> 'US, TX'
    - ', USRemote' -> 'US, Remote'
    - ', US, MSnull' -> 'US, MS'
    - ', USnull' -> 'US'
    """
    if pd.isna(loc_str) or not loc_str:
        return 'US'
    
    # Remove leading comma and spaces
    cleaned = loc_str.strip().lstrip(',').strip()
    
    # Check for Remote
    if 'Remote' in cleaned:
        return 'US, Remote'
    
    # Check for just "USnull" (no state)
    if cleaned == 'USnull':
        return 'US'
    
    # Pattern: "US, STATEcity" - extract state code (2 letters after "US, ")
    if ', ' in cleaned:
        parts = cleaned.split(', ')
        if len(parts) >= 2:
            state_city = parts[1]
            # Extract first 2 characters as state code
            state = state_city[:2].upper()
            return f'US, {state}'
    
    return 'US'

# Apply the cleaning function
df['location'] = df['location'].apply(clean_location)

# Check unique values
df['location'].unique()

<StringArray>
[    'US, TX',     'US, NJ',     'US, DC',     'US, CA',     'US, CO',
     'US, VA',     'US, WA',     'US, MN',     'US, UT',     'US, FL',
     'US, GA',     'US, MA',     'US, IL',     'US, OH',     'US, WV',
     'US, AZ',     'US, SC',     'US, NC', 'US, Remote',         'US',
     'US, IN',     'US, MD',     'US, MI',     'US, NY',     'US, KY',
     'US, ME',     'US, NV',     'US, NM',     'US, PA',     'US, OR',
     'US, NH',     'US, CT',     'US, MO',     'US, WI',     'US, OK',
     'US, TN',     'US, KS',     'US, LA',     'US, RI',     'US, NE',
     'US, ND',     'US, IA',     'US, MT',     'US, VT',     'US, ID',
     'US, AL',     'US, WY',     'US, MS']
Length: 48, dtype: str

## 3. Clean Bool Cols